# Colab GPU / TPU smoke (manual)

This notebook is **not** pytest and **not** CI. Run it by hand on
Google Colab before a release: once on a **GPU** runtime and once
on a **TPU** runtime.

It installs `kpnn2` from **TestPyPI** and leaves Colab's PyTorch
(CUDA or XLA) alone.

Open it from GitHub with the link in `dev/colab.txt`. Do not
upload a local copy.

## Runtime click

**Runtime → Change runtime type**, then pick one:

- GPU check: **T4 GPU**
- TPU check: **TPU v2** (not the deprecated TPU Node runtime)

Run all cells. The notebook detects CUDA vs XLA and skips the
checks that do not apply (for example `float16` on TPU).


## Install from TestPyPI

`--no-deps` is required so pip does not replace Colab's CUDA or
XLA torch with a CPU wheel. `xarray` is a core `kpnn2`
dependency and may be missing on Colab.


In [ ]:
# TestPyPI only. --no-deps keeps Colab's CUDA/XLA torch.
# --pre is required for release candidates (0.1.0rc1, ...).
%pip install -q --no-deps --pre -i https://test.pypi.org/simple/ kpnn2
%pip install -q xarray

# Unreleased SHA when TestPyPI is behind this commit:
# %pip install -q --no-deps \
#     git+https://github.com/Thomas-Rauter/kpnn2.git@main


## Detect the accelerator

If this cell fails, the runtime is still CPU. Change the runtime
type, then re-run from this cell (skip install if `kpnn2` is
already present).


In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

import kpnn2

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


def _tpu_device():
    try:
        import torch_xla
    except ImportError:
        return None
    device_fn = getattr(torch_xla, "device", None)
    if callable(device_fn):
        try:
            candidate = device_fn()
        except Exception:
            candidate = None
        else:
            if (
                isinstance(candidate, torch.device)
                and candidate.type == "xla"
            ):
                return candidate
    try:
        import torch_xla.core.xla_model as xm

        candidate = xm.xla_device()
    except Exception:
        return None
    if (
        isinstance(candidate, torch.device)
        and candidate.type == "xla"
    ):
        return candidate
    return None


if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
else:
    DEVICE = _tpu_device()

assert DEVICE is not None, (
    "No GPU or TPU. Use Runtime → Change runtime type "
    "and pick T4 GPU or TPU v2, then re-run this cell."
)

print("kpnn2", kpnn2.__version__)
print("torch", torch.__version__)
print("cuda runtime", torch.version.cuda)
print("device", DEVICE)
if DEVICE.type == "cuda":
    print("gpu", torch.cuda.get_device_name(0))
if DEVICE.type == "xla":
    print("PJRT_DEVICE", os.environ.get("PJRT_DEVICE"))


In [ ]:
def sync() -> None:
    if DEVICE.type != "xla":
        return
    import torch_xla.core.xla_model as xm

    xm.mark_step()


def to_cpu(tensor: torch.Tensor) -> torch.Tensor:
    sync()
    return tensor.detach().cpu()


def optimizer_step(optimizer) -> None:
    if DEVICE.type != "xla":
        optimizer.step()
        return
    import torch_xla.core.xla_model as xm

    step = getattr(xm, "optimizer_step", None)
    if callable(step):
        step(optimizer)
        return
    optimizer.step()
    xm.mark_step()


EDGELIST = pd.DataFrame(
    {
        "source": ["A", "B", "H", "A"],
        "target": ["H", "H", "C", "C"],
    }
)
SPEC = kpnn2.parse_layered(EDGELIST)
ADJ = kpnn2.parse_adjacency(EDGELIST)
FEATURES = pd.DataFrame(
    {
        "A": [0.1, 0.2, 0.3, 0.4],
        "B": [0.5, 0.6, 0.7, 0.8],
    }
)


class LayeredNet(nn.Module):
    def __init__(
        self,
        spec: kpnn2.LayeredSpec,
    ) -> None:
        super().__init__()
        self.spec = spec
        self.hops = nn.ModuleList(
            [
                kpnn2.MaskedLinear(hop.mask)
                for hop in spec.hops
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        saved = {0: x}
        last = len(self.hops) - 1
        hidden = x
        for index, hop in enumerate(self.spec.hops):
            sources = kpnn2.gather_hop_inputs(
                saved,
                hop,
            )
            hidden = self.hops[index](sources)
            if index < last:
                hidden = F.relu(hidden)
            saved[hop.target_layer] = hidden
        return hidden


print("helpers ready")


## MaskedLinear on the accelerator

Mask stays float32; parameters and the forward result follow the
module device. `mask_digest` stays a CPU `uint8` vector of length
32.


In [ ]:
torch.manual_seed(42)
layer = kpnn2.MaskedLinear(SPEC.hops[0].mask)
layer = layer.to(DEVICE)
assert isinstance(layer, kpnn2.MaskedLinear)
assert layer.mask.dtype == torch.float32
assert type(layer.mask) is torch.Tensor
assert layer.mask.device.type == DEVICE.type
original = layer.parametrizations.weight.original
assert original.device.type == DEVICE.type

x = kpnn2.align_inputs(FEATURES, SPEC).to(DEVICE)
y = layer(x)
assert y.device.type == DEVICE.type
assert y.shape == (4, 1)
loss = y.sum()
loss.backward()
assert original.grad is not None

digest = layer.state_dict()["mask_digest"]
assert digest.dtype == torch.uint8
assert digest.shape == (32,)
assert digest.device.type == "cpu"
assert "mask" not in layer.state_dict()

if DEVICE.type == "cuda":
    cpu_layer = kpnn2.MaskedLinear(SPEC.hops[0].mask)
    cpu_layer.load_state_dict(layer.state_dict())
    torch.testing.assert_close(
        to_cpu(layer(x)),
        cpu_layer(x.cpu()),
        atol=1e-5,
        rtol=1e-5,
    )

print("MaskedLinear move/forward/backward: ok")


## Dtype casts

`bfloat16` runs on both GPU and TPU. `float16` is CUDA-only
(TPU v2 is a bfloat16 machine). The stored mask stays float32.
Packed index buffers stay int64.


In [ ]:
def check_cast(
    apply_cast,
    expected_param_dtype,
) -> None:
    torch.manual_seed(42)
    layer = kpnn2.MaskedLinear(SPEC.hops[0].mask)
    layer = layer.to(DEVICE)
    layer = apply_cast(layer)
    assert layer.mask.dtype == torch.float32
    assert layer.mask.device.type == DEVICE.type
    original = layer.parametrizations.weight.original
    assert original.dtype == expected_param_dtype
    x = torch.ones(
        2,
        SPEC.hops[0].mask.shape[1],
        dtype=original.dtype,
        device=DEVICE,
    )
    y = layer(x)
    assert y.dtype == expected_param_dtype
    assert y.device.type == DEVICE.type


check_cast(
    lambda module: module.to(dtype=torch.bfloat16),
    torch.bfloat16,
)
if DEVICE.type == "cuda":
    check_cast(
        lambda module: module.half(),
        torch.float16,
    )

n = len(ADJ.nodes)
packed = kpnn2.PackedLinear(
    ADJ.source_index,
    ADJ.target_index,
    n,
    n,
)
packed = packed.to(DEVICE)
packed = packed.to(dtype=torch.bfloat16)
assert packed.source_index.dtype == torch.int64
assert packed.target_index.dtype == torch.int64
assert packed.weight.dtype == torch.bfloat16
assert packed.source_index.device.type == DEVICE.type

if DEVICE.type == "cuda":
    packed_half = kpnn2.PackedLinear(
        ADJ.source_index,
        ADJ.target_index,
        n,
        n,
    )
    packed_half = packed_half.to(DEVICE).half()
    assert packed_half.source_index.dtype == torch.int64
    assert packed_half.weight.dtype == torch.float16

print("dtype casts: ok")


## PackedLinear on the accelerator

Forward is `index_add` on ordinary dense tensors. Indices stay
int64 after the device move. `index_digest` stays CPU.


In [ ]:
torch.manual_seed(42)
n = len(ADJ.nodes)
core = kpnn2.PackedLinear(
    ADJ.source_index,
    ADJ.target_index,
    n,
    n,
)
core = core.to(DEVICE)
assert core.source_index.dtype == torch.int64
assert core.target_index.dtype == torch.int64
assert core.source_index.device.type == DEVICE.type
assert core.weight.device.type == DEVICE.type

x_in = kpnn2.align_inputs(FEATURES, ADJ).to(DEVICE)
state = torch.zeros(
    x_in.shape[0],
    n,
    dtype=x_in.dtype,
    device=DEVICE,
)
state[:, ADJ.input_index] = x_in
out = core(state)
assert out.device.type == DEVICE.type
assert out.shape == state.shape
out.sum().backward()
assert core.weight.grad is not None

digest = core.state_dict()["index_digest"]
assert digest.dtype == torch.uint8
assert digest.shape == (32,)
assert digest.device.type == "cpu"

print("PackedLinear move/forward/backward: ok")


## gather_hop_inputs device rules

All saved source layers must share a device. A CPU tensor mixed
with an accelerator tensor must raise `Kpnn2Error`.


In [ ]:
hop = SPEC.hops[1]
saved = {
    0: torch.ones(
        2,
        2,
        device=DEVICE,
    ),
    1: torch.ones(
        2,
        1,
        device=DEVICE,
    ),
}
gathered = kpnn2.gather_hop_inputs(
    saved,
    hop,
)
assert gathered.device.type == DEVICE.type
assert gathered.shape == (2, 3)

mixed = {
    0: torch.ones(
        2,
        2,
        device=DEVICE,
    ),
    1: torch.ones(
        2,
        1,
    ),
}
try:
    kpnn2.gather_hop_inputs(
        mixed,
        hop,
    )
except kpnn2.Kpnn2Error as exc:
    assert "device" in str(exc).lower()
else:
    raise AssertionError(
        "mixed CPU/accelerator gather should raise Kpnn2Error"
    )

print("gather_hop_inputs device: ok")


## One training step and named outputs

A tiny Adam step must change weights. `map_node_attributions`
must accept an accelerator tensor and return CPU values.


In [ ]:
torch.manual_seed(42)
model = LayeredNet(SPEC).to(DEVICE)
x = kpnn2.align_inputs(FEATURES, SPEC).to(DEVICE)
opt = torch.optim.Adam(
    model.parameters(),
    lr=0.05,
)
before = [
    p.detach().clone()
    for p in model.parameters()
]
opt.zero_grad()
loss = model(x).pow(2).mean()
loss.backward()
optimizer_step(opt)
sync()
moved = False
for left, right in zip(
    before,
    model.parameters(),
):
    if not torch.equal(
        to_cpu(left),
        to_cpu(right),
    ):
        moved = True
        break
assert moved, "expected a training step to change weights"

logits = model(x)
da = kpnn2.map_node_attributions(
    attributions=logits.detach(),
    spec=SPEC,
    layer=len(SPEC.layer_nodes) - 1,
)
assert list(da["node"].values) == list(
    SPEC.layer_nodes[-1]
)

print("training step + map_node_attributions: ok")


## torch.compile

On CUDA this uses the default backend (Inductor/Triton) with
`fullgraph=True`. That is stricter than the CPU tests, which use
`backend="eager"`.

On TPU it tries `backend="openxla"`. A skip here is printed,
not a failure: XLA compile support varies by Colab image.


In [ ]:
torch.manual_seed(42)
mask = SPEC.hops[0].mask
layer = kpnn2.MaskedLinear(mask).to(DEVICE)
x = torch.randn(
    4,
    mask.shape[1],
    device=DEVICE,
)
expected = to_cpu(layer(x))


def run_compile(
    module,
    data,
    backend=None,
):
    if backend is None:
        compiled = torch.compile(
            module,
            fullgraph=True,
        )
    else:
        compiled = torch.compile(
            module,
            fullgraph=True,
            backend=backend,
        )
    return to_cpu(compiled(data))


if DEVICE.type == "cuda":
    dynamo = torch._dynamo
    dynamo.reset()
    out = run_compile(
        layer,
        x,
    )
    torch.testing.assert_close(
        out,
        expected,
        atol=1e-4,
        rtol=1e-4,
    )
    packed = kpnn2.PackedLinear(
        ADJ.source_index,
        ADJ.target_index,
        len(ADJ.nodes),
        len(ADJ.nodes),
        bias=False,
    ).to(DEVICE)
    px = torch.randn(
        4,
        len(ADJ.nodes),
        device=DEVICE,
    )
    packed_expected = to_cpu(packed(px))
    dynamo.reset()
    packed_out = run_compile(
        packed,
        px,
    )
    torch.testing.assert_close(
        packed_out,
        packed_expected,
        atol=1e-4,
        rtol=1e-4,
    )
    print("torch.compile fullgraph (CUDA): ok")
elif DEVICE.type == "xla":
    try:
        out = run_compile(
            layer,
            x,
            backend="openxla",
        )
        torch.testing.assert_close(
            out,
            expected,
            atol=1e-3,
            rtol=1e-3,
        )
        print("torch.compile fullgraph (openxla): ok")
    except Exception as exc:
        print(
            "SKIP torch.compile on TPU:",
            type(exc).__name__,
            str(exc)[:200],
        )


In [ ]:
print()
print("ALL ACCELERATOR CHECKS PASSED")
print("device:", DEVICE)
print("kpnn2:", kpnn2.__version__)
